# Motor metrics — offline exploration

Continuous metrics for the four standardized trials, computed from recorded sessions.

| Trial | GMFM dim | What is measured underneath the ordinal item |
|---|---|---|
| `sit_hold` | B | uncensored duration; trunk lean; postural sway |
| `transition` | B | duration; SPARC smoothness; submovement count |
| `crawl` | C | cadence; cycle variability; left–right reciprocity (arms **and** legs) |
| `stand_hold` | D | same static-hold machinery as sitting |

**The premise.** GMFM items score 0–3 and AIMS is observed/not-observed. Remy can sit at a
"2" for a year while genuinely improving, and the score will not move. The item's job here is
to define a *reproducible trial*; this notebook measures the continuous variable underneath
it — the one that should move first.

**No external ground truth.** No PT-scored GMFM series exists for these sessions, so every
number is meaningful only against Remy's *own* baseline. Absolute values are not comparable
to published figures — see the SPARC caveat in `motor_metrics/transition.py`.

## Workflow

1. `pixi run record --output sessions/2026-07-16.h5`
2. `pixi run annotate sessions/2026-07-16.h5` — mark trials with the label vocabulary
3. run this notebook

Labels are `exercise[;key=value]*`, e.g. `sit_hold;arms=free;support=none;gmfm=23`. See
`motor_metrics/labels.py` for the full vocabulary.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # import the repo packages from notebooks/

from motor_metrics.labels import label_warnings
from motor_metrics.quality import TORSO, Gate, landmarks_ok
from motor_metrics.report import metrics_table, session_table
from motor_metrics.segments import segments
from motor_metrics.signals import (
    WORLD_UP,
    estimate_up,
    project_horizontal,
    trunk_from_vertical,
    trunk_vector,
)
from recording.reader import Recording

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SESSION = Path("../recording.hdf5")  # <- point this at a real session
SESSION

## 1. Label inventory

What did we actually mark, and does any of it have a typo? A mistyped value (`arms=freee`)
does not raise — it quietly becomes its own group in a `groupby` and splits a baseline in
half. Fix anything listed here in `annotate` before reading the numbers.

In [ ]:
rec = Recording(SESSION)

inventory = pd.DataFrame(
    [
        {
            "label": a.label,
            "start_s": a.start_ms / 1000,
            "end_s": a.end_ms / 1000,
            "dur_s": (a.end_ms - a.start_ms) / 1000,
            "warnings": "; ".join(label_warnings(a.label)),
        }
        for a in rec.annotations
    ]
)
print(f"{len(rec)} frames, {len(rec.annotations)} annotations, "
      f"{len(segments(rec))} usable trials")
inventory

## 2. Coverage QC

MediaPipe does not drop an occluded landmark — it **extrapolates** one and marks it
low-visibility. Those frames pass `pose_present` carrying invented coordinates, so every
metric gates on visibility and reports its coverage.

**A trial below ~0.8 coverage is not a measurement.** Check these before anything else.

In [ ]:
ok = landmarks_ok(rec, TORSO, Gate())
print(f"torso tracked on {ok.mean():.1%} of all frames")

fig, ax = plt.subplots(figsize=(12, 1.6))
ax.fill_between(rec.timestamps_ms / 1000, 0, ok.astype(int), step="mid", alpha=0.6)
for seg in segments(rec):
    ax.axvspan(seg.ann.start_ms / 1000, seg.ann.end_ms / 1000, color="tab:orange", alpha=0.2)
    ax.text(seg.ann.start_ms / 1000, 1.05, seg.exercise, fontsize=7, rotation=30)
ax.set(xlabel="session time (s)", ylabel="tracked", yticks=[0, 1], ylim=(0, 1.4))
ax.set_title("Torso tracking (bars) vs marked trials (shaded)")
plt.tight_layout()

## 3. Vertical-reference diagnostic — run this before trusting any hold number

There is no gravity vector in a camera-only recording. `WORLD_UP` is the camera's own
y-axis, so **it is vertical only if the camera was level**. If it was not, every trunk-lean
and sway number is biased by the tilt, and nothing else will tell you.

Mark a `calib;pose=upright` segment (Remy held upright, camera as-placed) and check that the
median trunk angle below is near 0. If it is not, either level the camera and re-record, or
pass `up=estimate_up(...)` below — but read the caveat first: a calibration pose cannot
separate *camera tilt* from *a child who does not sit vertically*, and for Remy that second
term is not small. Calibrating on it can bake a real postural asymmetry into the reference
and hide it permanently. Prefer a level camera.

In [ ]:
calib = segments(rec, "calib")
up = WORLD_UP

if not calib:
    print("No `calib;pose=upright` segment. Assuming a level camera (up = world -y).")
    print("Mark one in `annotate` to check that assumption.")
else:
    angles = trunk_from_vertical(rec.landmarks_world, up=WORLD_UP)
    seg = calib[0]
    tilt = np.nanmedian(angles[seg.start : seg.stop])
    print(f"median trunk angle over the calibration pose: {tilt:.1f} deg")
    if tilt > 10:
        print("  -> more than 10 deg off vertical. Either the camera is tilted or Remy")
        print("     is not upright in the calibration pose -- this cannot tell which.")
        print("     Uncomment below only if you know it is the camera:")
        print("     # up = estimate_up(rec, seg)")
    else:
        print("  -> consistent with a level camera; using world -y.")

print(f"using up = {up}")

## 4. The metric table

One row per trial. Columns are the union across exercise types, so a sitting row has NaN in
the crawl columns and vice versa — that is what lets them concatenate into a trend.

Nothing here is written back into the `.h5`: metrics are recomputed on read, so the filter
constants stay changeable and the raw landmarks stay the record.

In [ ]:
table = metrics_table(rec, up=up, session=SESSION.stem)
table

## 5. Holds — sitting and supported standing

`duration_s` is the headline: the uncensored version of the GMFM 3/5/20/60-second ladder.

Two things to keep straight when reading the sway columns:

- **`path_length_m` is confounded with duration** — the same sway measured for longer
  travels further. Compare it only at equal `window_s`; otherwise use `mean_velocity_mps`.
- **ML is trustworthy, AP is not.** ML is lateral, in the image plane. AP is MediaPipe's
  inferred depth on a single camera and is markedly noisier. Point the camera so the sway
  you care about is lateral.

In [ ]:
holds = table[table["exercise"].isin(["sit_hold", "stand_hold"])]
cols = ["exercise", "p_arms", "p_support", "duration_s", "coverage",
        "mean_velocity_mps", "ellipse_area_m2", "sway_ml_rms_m", "sway_ap_rms_m",
        "trunk_angle_mean_deg", "trunk_angle_range_deg"]
holds[[c for c in cols if c in holds.columns]]

In [ ]:
# Sway paths: the actual trunk-over-pelvis excursion behind those numbers.
hold_segs = [s for s in segments(rec) if s.exercise in ("sit_hold", "stand_hold")]

if hold_segs:
    fig, axes = plt.subplots(
        1, len(hold_segs), figsize=(3.2 * len(hold_segs), 3.2), squeeze=False
    )
    for ax, seg in zip(axes[0], hold_segs):
        tip = trunk_vector(rec.landmarks_world)[seg.start : seg.stop]
        h = project_horizontal(tip, up=up)
        h = h[np.isfinite(h).all(axis=1)]
        h = h - h.mean(axis=0)
        ax.plot(h[:, 0] * 100, h[:, 1] * 100, lw=0.7, alpha=0.8)
        ax.set(xlabel="ML (cm)", ylabel="AP (cm)", title=seg.exercise)
        ax.axhline(0, lw=0.5, color="grey")
        ax.axvline(0, lw=0.5, color="grey")
        ax.set_aspect("equal")
    plt.tight_layout()
else:
    print("No hold trials marked in this session.")

## 6. Transitions — smoothness

`sparc_trunk` is the primary: more negative = less smooth. **Read it only against other
values from this same pipeline** — at 30 Hz with landmark noise the absolute number is a
property of the filter chain, not of Remy.

It is also only reliable when the speed profile is clean against its own peak (robust to
~2 % noise, erratic past ~5 %). Big brisk transitions score reliably; small slow ones may
not. Prefer the median of several trials to any single one.

In [ ]:
trans = table[table["exercise"] == "transition"]
cols = ["p_from", "p_to", "p_side", "duration_s", "movement_duration_s", "coverage",
        "sparc_trunk", "n_velocity_peaks", "peak_angular_velocity_dps", "leading_wrist"]
trans[[c for c in cols if c in trans.columns]]

In [ ]:
# Side symmetry is a BETWEEN-trial comparison: a single transition happens to one side,
# so asking whether *it* was symmetric is not a question. Needs trials on both sides.
from motor_metrics.transition import symmetry_index

if not trans.empty and "p_side" in trans.columns:
    for metric in ("movement_duration_s", "sparc_trunk"):
        left = trans.loc[trans["p_side"] == "left", metric]
        right = trans.loc[trans["p_side"] == "right", metric]
        si = symmetry_index(left, right)
        print(f"{metric:24s} symmetry index: {si:+.3f}"
              f"   (n_left={len(left)}, n_right={len(right)})")
    print("\n0 = both sides alike; sign points at the larger side. NaN = one side unmarked.")
else:
    print("No side-labelled transitions in this session.")

## 7. Crawl — cadence and reciprocity, arms *and* legs

**`speed_norm_per_s` is in image widths per second, not metres.** The world frame is
hip-centered, so travel across the floor is not in it, and the GMFM item's "1.8 m" is not
measurable here. That column is comparable only within one session at a fixed camera.

The real deliverable is the pattern, and it is measured at **both limb girdles** — arms
(wrists, the unprefixed columns) and legs (knees, the `leg_*` columns):

- `phase_offset` / `leg_phase_offset` — **0.5 = reciprocal** (the pair alternating, the
  mature pattern), **0.0 = symmetric** (both moving together, a "bunny" haul)
- `cycle_period_cv` / `leg_cycle_period_cv` — how metronomic the cycles are
- `amplitude_symmetry` / `leg_amplitude_symmetry` — whether one limb does more of the
  work; for the legs this is the **"favors one leg"** reading (0 = even, sign = side)

**Read both girdles — Remy's signal is in the legs.** His arms often move together while he
drives with the legs and favors one repeatedly, so that pattern shows as a low `phase_offset`
(arms hauling together) next to a nonzero `leg_amplitude_symmetry` (a favored leg). Each
girdle carries its own `coverage`/`tracked_s` (`leg_coverage`/`leg_tracked_s`): legs leave
frame in prone more than arms, so a low `leg_coverage` next to a fine `coverage` means the
leg numbers, not the crawl, are untrustworthy.

In [ ]:
crawls = table[table["exercise"] == "crawl"]
arm_cols = ["p_dir", "duration_s", "coverage", "cadence_cpm", "cadence_cpm_left",
            "cadence_cpm_right", "cycle_period_cv", "phase_offset",
            "amplitude_symmetry", "speed_norm_per_s"]
leg_cols = ["p_dir", "leg_coverage", "leg_cadence_cpm", "leg_cadence_cpm_left",
            "leg_cadence_cpm_right", "leg_cycle_period_cv", "leg_phase_offset",
            "leg_amplitude_symmetry"]
print("Arms (wrists):")
display(crawls[[c for c in arm_cols if c in crawls.columns]])
print("Legs (knees) \u2014 leg_amplitude_symmetry is the 'favors one leg' reading (0 = even, sign = side):")
crawls[[c for c in leg_cols if c in crawls.columns]]

In [ ]:
# The limb signals themselves: each limb's reach along the body axis. Alternating humps
# are a reciprocal crawl; humps in lockstep are a symmetric haul; one flat side is a
# favored limb -- read the legs (bottom) as much as the arms (top).
from motor_metrics.crawl import limb_signal

crawl_segs = segments(rec, "crawl")
if crawl_segs:
    seg = crawl_segs[0]
    t = rec.timestamps_ms[seg.start : seg.stop] / 1000
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    for ax, marker, title in ((axes[0], "wrist", "arms (wrists)"),
                              (axes[1], "knee", "legs (knees)")):
        for side, colour in (("left", "tab:blue"), ("right", "tab:red")):
            ax.plot(t, limb_signal(rec, seg, side, marker) * 100,
                    color=colour, label=side, lw=1)
        ax.set(ylabel="reach along\nbody axis (cm)",
               title=f"Belly-crawl limb signals \u2014 {title}")
        ax.legend(loc="upper right")
    axes[1].set_xlabel("session time (s)")
    plt.tight_layout()
else:
    print("No crawl trials marked in this session.")

## 8. Cross-session trend — the point of all of it

With no external GMFM score to calibrate against, a single session's number means little.
The signal is the *trend* against Remy's own baseline: the thing that should move before
the ordinal item does.

Two rules for keeping this honest over months:

- **Do not change the constants in `motor_metrics/derive.py`.** Every number here is a
  function of them; changing one silently rebases the whole history.
- **Keep the camera placed the same way.** `up_source` records the assumption per row.

In [ ]:
sessions = sorted(Path("../sessions").glob("*.h5")) if Path("../sessions").exists() else []

if len(sessions) < 2:
    print("Need >= 2 recordings in ../sessions/ for a trend. Record more.")
else:
    trend = session_table(sessions)
    sits = trend[(trend["exercise"] == "sit_hold") & (trend["coverage"] > 0.8)]

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, metric, label in zip(
        axes,
        ["duration_s", "mean_velocity_mps", "ellipse_area_m2"],
        ["hold duration (s)", "mean sway velocity (m/s)", "95% sway ellipse (m^2)"],
    ):
        by_session = sits.groupby("session")[metric]
        ax.errorbar(
            range(len(by_session)), by_session.median(), yerr=by_session.std(),
            marker="o", capsize=3,
        )
        ax.set_xticks(range(len(by_session)))
        ax.set_xticklabels(by_session.groups.keys(), rotation=45, ha="right")
        ax.set_ylabel(label)
    axes[0].set_title("Longer is better")
    axes[1].set_title("Lower is steadier")
    axes[2].set_title("Smaller is steadier")
    plt.tight_layout()
    display(sits.groupby("session")[["duration_s", "mean_velocity_mps"]].median())

In [ ]:
rec.close()